# Pepper Inference

Colab-friendly notebook for loading a Textual Inversion or DreamBooth
checkpoint produced by this repo and sampling Pepper images from prompts
with the manual DDIM sampler defined in `src/utils/sampling.py`.

**Important**: this notebook only *uses* the building blocks in `src/`; it
does not introduce a pipeline wrapper. Every diffusion step is still run
through our hand-written sampler.

## 0. Colab bootstrap (skip locally)

In [ ]:
RUNNING_IN_COLAB = False
try:
    import google.colab  # noqa: F401
    RUNNING_IN_COLAB = True
except ImportError:
    pass

if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPOSITORY_ROOT = '/content/drive/MyDrive/persistent-pepper'
    !pip -q install -r {REPOSITORY_ROOT}/requirements.txt
else:
    from pathlib import Path
    REPOSITORY_ROOT = str(Path().resolve().parent)

print('Repository root:', REPOSITORY_ROOT)

## 1. Put the repo on `sys.path` and import building blocks

In [ ]:
import sys
sys.path.insert(0, REPOSITORY_ROOT)

import torch
from pathlib import Path
import matplotlib.pyplot as plt

from src.models import load_stable_diffusion_components
from src.utils.sampling import SamplingConfiguration, sample_images_from_prompts

pretrained_model_name_or_path = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
# Swap to a local dir if you've pre-downloaded:
# pretrained_model_name_or_path = f'{REPOSITORY_ROOT}/checkpoints/sd-v1-5'

stable_diffusion_components = load_stable_diffusion_components(
    pretrained_model_name_or_path=pretrained_model_name_or_path,
    working_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
print('Loaded SD 1.5 onto', stable_diffusion_components.working_device)

## 2a. Load a Textual Inversion checkpoint (skip if using DreamBooth)

In [ ]:
from scripts.evaluate import load_textual_inversion_embedding_into_components

learned_embedding_checkpoint_path = Path(
    REPOSITORY_ROOT
) / 'checkpoints' / 'textual_inversion' / 'learned_embedding_final.bin'

placeholder_token_string = load_textual_inversion_embedding_into_components(
    stable_diffusion_components=stable_diffusion_components,
    learned_embedding_checkpoint_path=learned_embedding_checkpoint_path,
)
print('Loaded textual inversion embedding:', placeholder_token_string)

subject_phrase_for_prompts = placeholder_token_string

## 2b. Load a DreamBooth checkpoint (skip if using Textual Inversion)

In [ ]:
from scripts.evaluate import load_dreambooth_weights_into_components

dreambooth_checkpoint_directory = Path(
    REPOSITORY_ROOT
) / 'checkpoints' / 'dreambooth' / 'latest'

load_dreambooth_weights_into_components(
    stable_diffusion_components=stable_diffusion_components,
    dreambooth_checkpoint_directory=dreambooth_checkpoint_directory,
)

subject_phrase_for_prompts = 'sks dog'

## 3. Sample images

In [ ]:
prompt_templates = [
    'a photo of {subject} wearing a red wool sweater',
    'an oil painting of {subject} in the style of Van Gogh',
    'a photo of {subject} on the surface of the moon',
    'a pencil sketch of {subject}',
]
fully_substituted_prompts = [
    prompt_template.format(subject=subject_phrase_for_prompts)
    for prompt_template in prompt_templates
]

sampling_configuration = SamplingConfiguration(
    number_of_inference_steps=50,
    classifier_free_guidance_scale=7.5,
    random_seed=42,
)

generated_pil_images = sample_images_from_prompts(
    stable_diffusion_components=stable_diffusion_components,
    prompt_texts=fully_substituted_prompts,
    sampling_configuration=sampling_configuration,
)

figure, axes_grid = plt.subplots(
    1,
    len(generated_pil_images),
    figsize=(4 * len(generated_pil_images), 4),
)
if len(generated_pil_images) == 1:
    axes_grid = [axes_grid]
for axis, generated_pil_image, fully_substituted_prompt in zip(
    axes_grid,
    generated_pil_images,
    fully_substituted_prompts,
):
    axis.imshow(generated_pil_image)
    axis.set_title(fully_substituted_prompt, fontsize=8)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 4. Optional: CLIP-based identity / prompt alignment report

In [ ]:
from src.utils.metrics import (
    compute_clip_similarity_report,
    format_report_as_human_readable_text,
    load_pil_images_from_directory,
)

reference_pil_images = load_pil_images_from_directory(
    Path(REPOSITORY_ROOT) / 'data' / 'processed'
)
clip_similarity_report = compute_clip_similarity_report(
    generated_pil_images=generated_pil_images,
    reference_pil_images=reference_pil_images,
    prompts_used_for_generation=fully_substituted_prompts,
)
print(format_report_as_human_readable_text(clip_similarity_report))